In [15]:
import os
import faiss
import numpy as np
import pandas as pd

import pickle

In [17]:
import tensorflow as tf

from collections import Counter

from tensorflow.keras.models import load_model

from tensorflow.keras.preprocessing.image import (
    load_img,
    img_to_array
)

from tensorflow.keras.applications.resnet50 import (
    preprocess_input
)

from tensorflow.keras.preprocessing.sequence import (
    pad_sequences
)

In [36]:
MODEL_PATH = "../../counterfeit_image_detection/models/embedding_model_finetuned.keras"

EMBEDDINGS_PATH = "../../counterfeit_image_detection/embeddings/shoe_image_embeddings_finetuned.npy"

FILENAMES_PATH = "../../counterfeit_image_detection/embeddings/shoe_image_filenames.npy"

FAISS_PATH = "../../counterfeit_image_detection/embeddings/shoe_faiss_index.index"

METADATA_MAP_PATH = "../../counterfeit_metadata_detection/datasets/authentic_metadata_output.csv"

TEST1_PATH = "../testing/test1_genuine_metadata.csv"

TEST2_PATH = "../testing/test2_genuine_metadata.csv"

TEST3_PATH = "../testing/test3_brand_mismatch.csv"

TEST4_PATH = "../testing/test4_counterfeit_metadata.csv"

SEEN_IMAGES_DIR = "../testing/SeenImages"

UNSEEN_IMAGES_DIR = "../testing/UnseenImages"

In [8]:
class L2Normalize(
    tf.keras.layers.Layer
):
    def call(
        self,
        inputs
    ):
        return tf.math.l2_normalize(
            inputs,
            axis=1
        )

embedding_model = load_model(
    MODEL_PATH,
    custom_objects={
        "L2Normalize":
        L2Normalize
    },
    compile=False
)

In [9]:
image_embeddings = np.load(
    EMBEDDINGS_PATH
)

image_filenames = np.load(
    FILENAMES_PATH,
    allow_pickle=True
)

print(
    image_embeddings.shape
)

print(
    len(image_filenames)
)

(2642, 128)
2642


In [10]:
image_index = faiss.read_index(
    FAISS_PATH
)

print(
    "Total indexed images:",
    image_index.ntotal
)

Total indexed images: 2642


In [11]:
metadata_df = pd.read_csv(
    METADATA_MAP_PATH
)

metadata_map = dict(
    zip(
        metadata_df["filename"],
        metadata_df["metadata"]
    )
)

print(
    len(metadata_map)
)

2642


In [12]:
def extract_brand(text):
    text = str(text).lower()

    if ("new balance" in text or "newbalance" in text):
        return "newbalance"

    if ("under armour" in text or "underarmour" in text):
        return "underarmour"

    brands = [
        "nike",
        "puma",
        "vans",
        "reebok"
    ]

    for brand in brands:

        if brand in text:
            return brand

    return "unknown"

In [13]:
def get_embedding(image_path):
    image = load_img(
        image_path,
        target_size=(224,224)
    )

    image = img_to_array(image)

    image = preprocess_input(image)

    image = np.expand_dims(
        image,
        axis=0
    )

    embedding = embedding_model.predict(
        image,
        verbose=0
    )

    return embedding.astype(
        "float32"
    )

In [14]:
def retrieve_similar_images(image_path, top_k=5):
    embedding = get_embedding(
        image_path
    )

    faiss.normalize_L2(embedding)

    similarities, indices = (
        image_index.search(
            embedding,
            top_k
        )
    )

    results = []

    for sim, idx in zip(similarities[0], indices[0]):
        results.append(
            {
                "filename":
                image_filenames[idx],

                "similarity":
                float(sim)
            }
        )

    return results

In [16]:
metadata_classifier = load_model(
    "../../counterfeit_metadata_detection/models/metadata_counterfeit_classifier_v1.keras",
    compile=False
)

with open(
    "../../counterfeit_metadata_detection/tokenizers/metadata_classifier_tokenizer_v1.pkl",
    "rb"
) as f:

    metadata_tokenizer = pickle.load(
        f
    )

In [18]:
MAX_LEN = 110

In [23]:
def get_metadata_score(metadata_text):
    sequence = (
        metadata_tokenizer.texts_to_sequences(
            [metadata_text]
        )
    )

    padded = pad_sequences(
        sequence,
        maxlen=MAX_LEN,
        padding="post"
    )

    prob_fake = float(
        metadata_classifier.predict(
            padded,
            verbose=0
        )[0][0]
    )

    prob_genuine = 1.0 - prob_fake
    
    return prob_genuine

In [20]:
def trust_fusion(query_metadata, retrieval_results):
    metadata_score = (
        get_metadata_score(
            query_metadata
        )
    )

    query_brand = extract_brand(
        query_metadata
    )

    retrieved_brands = []

    similarities = []

    for result in retrieval_results:
        similarities.append(
            result["similarity"]
        )

        retrieved_metadata = (
            metadata_map.get(
                result["filename"],
                ""
            )
        )

        retrieved_brand = (
            extract_brand(
                retrieved_metadata
            )
        )

        retrieved_brands.append(
            retrieved_brand
        )

    brand_match_ratio = (
        retrieved_brands.count(query_brand) / len(retrieved_brands)
    )

    retrieval_confidence = (
        np.mean(similarities)
    )

    trust_score = (
        0.50 * metadata_score
        +
        0.35 * brand_match_ratio
        +
        0.15 * retrieval_confidence
    )

    if trust_score >= 0.75:
        prediction = "GENUINE"

    elif trust_score >= 0.50:
        prediction = "SUSPICIOUS"

    else:
        prediction = (
            "LIKELY_COUNTERFEIT"
        )

    return {
        "prediction":
        prediction,

        "trust_score":
        round(
            trust_score,
            4
        ),

        "metadata_score":
        round(
            metadata_score,
            4
        ),

        "brand_match_ratio":
        round(
            brand_match_ratio,
            4
        ),

        "retrieval_confidence":
        round(
            retrieval_confidence,
            4
        ),

        "retrieved_brands":
        retrieved_brands
    }

In [28]:
test1_df = pd.read_csv(
    TEST1_PATH
)

print(
    test1_df.columns
)

test1_df.head()

Index(['filename', 'metadata'], dtype='object')


,filename,metadata
0,normalized_nike_11.jpg,Nike Air Force 1 Low Brown White Casual Sneakers
1,normalized_nike_249.jpg,Nike Dunk Low Red White Casual Sneakers
2,resized_all_vans_129.jpg,Vans Sport Low White Green Orange Skate Shoes
3,resized_all_vans_140.jpg,Vans UltraRange Grey Black Lime Running Shoes
4,resized_all_vans_17.jpg,Vans Sport Low Black White Skate Shoes


In [26]:
results = []
correct = 0

for _, row in test1_df.iterrows():
    image_file = row["filename"]

    query_metadata = row["metadata"]

    image_path = os.path.join(
        SEEN_IMAGES_DIR,
        image_file
    )

    retrieval_results = (
        retrieve_similar_images(
            image_path,
            top_k=5
        )
    )

    fusion_result = trust_fusion(
        query_metadata,
        retrieval_results
    )

    prediction = fusion_result["prediction"]

    is_correct = (prediction == "GENUINE")

    if is_correct:
        correct += 1

    results.append(
        {
            "filename":
            image_file,

            "prediction":
            prediction,

            "trust_score":
            fusion_result[
                "trust_score"
            ],

            "metadata_score":
            fusion_result[
                "metadata_score"
            ],

            "brand_match_ratio":
            fusion_result[
                "brand_match_ratio"
            ],

            "retrieval_confidence":
            fusion_result[
                "retrieval_confidence"
            ],

            "correct":
            is_correct
        }
    )

results_df = pd.DataFrame(
    results
)

accuracy = correct / len(results_df)

print("\nTest 1 Accuracy:", round(accuracy, 4))

results_df.head(20)


Test 1 Accuracy: 0.9524


,filename,prediction,trust_score,metadata_score,brand_match_ratio,retrieval_confidence,correct
0,normalized_nike_11.jpg,GENUINE,0.9943,1.0,1.0,0.9620,True
1,normalized_nike_249.jpg,GENUINE,0.9884,1.0,1.0,0.9227,True
2,resized_all_vans_129.jpg,GENUINE,0.9830,1.0,1.0,0.8868,True
3,resized_all_vans_140.jpg,GENUINE,0.9898,1.0,1.0,0.9321,True
4,resized_all_vans_17.jpg,GENUINE,0.9893,1.0,1.0,0.9284,True
5,resized_all_vans_32.jpg,GENUINE,0.9818,1.0,1.0,0.8785,True
6,resized_padded_new_newbalance_2.jpg,GENUINE,0.9846,1.0,1.0,0.8977,True
7,resized_padded_new_newbalance_3.jpg,GENUINE,0.9926,1.0,1.0,0.9507,True
8,resized_padded_new_newbalance_53.jpg,GENUINE,0.9919,1.0,1.0,0.9461,True
9,resized_padded_puma_135.jpg,GENUINE,0.9715,1.0,1.0,0.8103,True


In [29]:
test2_df = pd.read_csv(
    TEST2_PATH
)

print(
    test2_df.shape
)

test2_df.head()

(30, 2)


,filename,metadata
0,new balance-1.jpg,New Balance 997 Black Purple Grey Casual Sneakers
1,new balance-2.jpg,New Balance 990 Grey Suede Running Shoes
2,new balance-3.jpg,New Balance 991 Black Grey Purple Running Shoes
3,new balance-4.jpg,New Balance 990 Black Blue Grey Running Shoes
4,new balance-5.jpg,New Balance 550 White Red Black Casual Sneakers


In [30]:
results = []
correct = 0

for _, row in test2_df.iterrows():
    image_file = row["filename"]

    query_metadata = row["metadata"]

    image_path = os.path.join(
        UNSEEN_IMAGES_DIR,
        image_file
    )

    retrieval_results = (
        retrieve_similar_images(
            image_path,
            top_k=5
        )
    )

    fusion_result = trust_fusion(
        query_metadata,
        retrieval_results
    )

    prediction = fusion_result["prediction"]

    is_correct = (prediction == "GENUINE")

    if is_correct:
        correct += 1

    results.append(
        {
            "filename":
            image_file,

            "prediction":
            prediction,

            "trust_score":
            fusion_result[
                "trust_score"
            ],

            "metadata_score":
            fusion_result[
                "metadata_score"
            ],

            "brand_match_ratio":
            fusion_result[
                "brand_match_ratio"
            ],

            "retrieval_confidence":
            fusion_result[
                "retrieval_confidence"
            ],

            "retrieved_brands":
            fusion_result[
                "retrieved_brands"
            ],

            "correct":
            is_correct
        }
    )

results_df = pd.DataFrame(results)

accuracy = correct / len(results_df)

print("\nTest 2 Accuracy:", round(accuracy, 4))

results_df.head(20)


Test 2 Accuracy: 0.7667


,filename,prediction,trust_score,metadata_score,brand_match_ratio,retrieval_confidence,retrieved_brands,correct
0,new balance-1.jpg,SUSPICIOUS,0.6256,1.0000,0.0,0.8376,"[vans, vans, vans, vans, vans]",False
1,new balance-2.jpg,GENUINE,0.9777,1.0000,1.0,0.8515,"[newbalance, newbalance, newbalance, newbalanc...",True
2,new balance-3.jpg,SUSPICIOUS,0.6209,1.0000,0.0,0.8062,"[vans, vans, vans, vans, vans]",False
3,new balance-4.jpg,GENUINE,0.8956,1.0000,0.8,0.7706,"[newbalance, newbalance, vans, newbalance, new...",True
4,new balance-5.jpg,GENUINE,0.9802,1.0000,1.0,0.8681,"[newbalance, newbalance, newbalance, newbalanc...",True
5,nike-1.jpg,GENUINE,0.9793,0.9999,1.0,0.8624,"[nike, nike, nike, nike, nike]",True
6,nike-2.jpg,GENUINE,0.9129,1.0000,0.8,0.8863,"[nike, puma, nike, nike, nike]",True
7,nike-3.jpg,GENUINE,0.9785,1.0000,1.0,0.8565,"[nike, nike, nike, nike, nike]",True
8,nike-4.jpg,SUSPICIOUS,0.6017,1.0000,0.0,0.6781,"[puma, puma, puma, puma, puma]",False
9,nike-5.jpg,SUSPICIOUS,0.7009,0.9996,0.2,0.8742,"[puma, puma, puma, nike, puma]",False


In [31]:
failures = results_df[
    results_df["correct"] == False
]

print("\nFailures:",len(failures))

failures


Failures: 7


,filename,prediction,trust_score,metadata_score,brand_match_ratio,retrieval_confidence,retrieved_brands,correct
0,new balance-1.jpg,SUSPICIOUS,0.6256,1.0000,0.0,0.8376,"[vans, vans, vans, vans, vans]",False
2,new balance-3.jpg,SUSPICIOUS,0.6209,1.0000,0.0,0.8062,"[vans, vans, vans, vans, vans]",False
8,nike-4.jpg,SUSPICIOUS,0.6017,1.0000,0.0,0.6781,"[puma, puma, puma, puma, puma]",False
9,nike-5.jpg,SUSPICIOUS,0.7009,0.9996,0.2,0.8742,"[puma, puma, puma, nike, puma]",False
14,resized_padded_puma_210.jpg,SUSPICIOUS,0.6190,1.0000,0.0,0.7932,"[underarmour, underarmour, underarmour, undera...",False
17,resized_padded_reebok_156.jpg,SUSPICIOUS,0.6353,1.0000,0.0,0.9021,"[vans, vans, vans, vans, vans]",False
21,resized_padded_reebok_73.jpg,SUSPICIOUS,0.6748,1.0000,0.2,0.6984,"[vans, vans, reebok, vans, vans]",False


In [33]:
test3_df = pd.read_csv(
    TEST3_PATH
)

print(
    test3_df.shape
)

test3_df.head()

(30, 2)


,filename,mismatched_metadata
0,new balance-1.jpg,Nike Air Max 90 Black Purple Grey Casual Sneakers
1,new balance-2.jpg,Puma Suede Classic Grey White Casual Sneakers
2,new balance-3.jpg,ASICS Gel Lyte Black Grey Purple Running Shoes
3,new balance-4.jpg,Reebok Classic Nylon Black Blue Grey Casual Sn...
4,new balance-5.jpg,Nike Dunk Low White Red Black Casual Sneakers


In [35]:
results = []
correct = 0

for _, row in test3_df.iterrows():
    image_file = row["filename"]

    query_metadata = row["mismatched_metadata"]

    image_path = os.path.join(
        UNSEEN_IMAGES_DIR,
        image_file
    )

    retrieval_results = (
        retrieve_similar_images(
            image_path,
            top_k=5
        )
    )

    fusion_result = trust_fusion(
        query_metadata,
        retrieval_results
    )

    prediction = fusion_result["prediction"]

    is_correct = (prediction != "GENUINE")

    if is_correct:
        correct += 1

    results.append(
        {
            "filename":
            image_file,

            "prediction":
            prediction,

            "trust_score":
            fusion_result[
                "trust_score"
            ],

            "metadata_score":
            fusion_result[
                "metadata_score"
            ],

            "brand_match_ratio":
            fusion_result[
                "brand_match_ratio"
            ],

            "retrieval_confidence":
            fusion_result[
                "retrieval_confidence"
            ],

            "retrieved_brands":
            fusion_result[
                "retrieved_brands"
            ],

            "correct":
            is_correct
        }
    )

results_df = pd.DataFrame(
    results
)

accuracy = correct / len(results_df)

print("\nTest 3 Accuracy:", round(accuracy, 4))

results_df


Test 3 Accuracy: 0.9


,filename,prediction,trust_score,metadata_score,brand_match_ratio,retrieval_confidence,retrieved_brands,correct
0,new balance-1.jpg,SUSPICIOUS,0.6256,0.9999,0.0,0.8376,"[vans, vans, vans, vans, vans]",True
1,new balance-2.jpg,SUSPICIOUS,0.6277,1.0000,0.0,0.8515,"[newbalance, newbalance, newbalance, newbalanc...",True
2,new balance-3.jpg,SUSPICIOUS,0.6209,1.0000,0.0,0.8062,"[vans, vans, vans, vans, vans]",True
3,new balance-4.jpg,SUSPICIOUS,0.6156,1.0000,0.0,0.7706,"[newbalance, newbalance, vans, newbalance, new...",True
4,new balance-5.jpg,SUSPICIOUS,0.6302,1.0000,0.0,0.8681,"[newbalance, newbalance, newbalance, newbalanc...",True
5,nike-1.jpg,SUSPICIOUS,0.6294,1.0000,0.0,0.8624,"[nike, nike, nike, nike, nike]",True
6,nike-2.jpg,SUSPICIOUS,0.6329,1.0000,0.0,0.8863,"[nike, puma, nike, nike, nike]",True
7,nike-3.jpg,SUSPICIOUS,0.6285,1.0000,0.0,0.8565,"[nike, nike, nike, nike, nike]",True
8,nike-4.jpg,SUSPICIOUS,0.6017,1.0000,0.0,0.6781,"[puma, puma, puma, puma, puma]",True
9,nike-5.jpg,SUSPICIOUS,0.6311,1.0000,0.0,0.8742,"[puma, puma, puma, nike, puma]",True


In [37]:
test4_df = pd.read_csv(
    TEST4_PATH
)

print(
    test4_df.shape
)

test4_df.head()

(30, 2)


,filename,counterfeit_metadata
0,new balance-1.jpg,New Balance 997 Black Purple Grey Premium Copy...
1,new balance-2.jpg,New Balance 990 Grey Suede UA Quality Running ...
2,new balance-3.jpg,New Balance 991 Black Grey Purple Imported Cop...
3,new balance-4.jpg,New Balance 990 Black Blue Grey Mirror Quality...
4,new balance-5.jpg,New Balance 550 White Red Black Replica Casual...


In [39]:
results = []
correct = 0

for _, row in test4_df.iterrows():
    image_file = row["filename"]

    query_metadata = row["counterfeit_metadata"]

    image_path = os.path.join(
        UNSEEN_IMAGES_DIR,
        image_file
    )

    retrieval_results = (
        retrieve_similar_images(
            image_path,
            top_k=5
        )
    )

    fusion_result = trust_fusion(
        query_metadata,
        retrieval_results
    )

    prediction = fusion_result["prediction"]

    is_correct = (prediction != "GENUINE")

    if is_correct:
        correct += 1

    results.append(
        {
            "filename":
            image_file,

            "prediction":
            prediction,

            "trust_score":
            fusion_result[
                "trust_score"
            ],

            "metadata_score":
            fusion_result[
                "metadata_score"
            ],

            "brand_match_ratio":
            fusion_result[
                "brand_match_ratio"
            ],

            "retrieval_confidence":
            fusion_result[
                "retrieval_confidence"
            ],

            "retrieved_brands":
            fusion_result[
                "retrieved_brands"
            ],

            "correct":
            is_correct
        }
    )

results_df = pd.DataFrame(
    results
)

accuracy = (correct / len(results_df))

print("\nTest 4 Accuracy:", round(accuracy, 4))

results_df


Test 4 Accuracy: 0.9333


,filename,prediction,trust_score,metadata_score,brand_match_ratio,retrieval_confidence,retrieved_brands,correct
0,new balance-1.jpg,LIKELY_COUNTERFEIT,0.1256,0.0000,0.0,0.8376,"[vans, vans, vans, vans, vans]",True
1,new balance-2.jpg,LIKELY_COUNTERFEIT,0.4777,0.0000,1.0,0.8515,"[newbalance, newbalance, newbalance, newbalanc...",True
2,new balance-3.jpg,LIKELY_COUNTERFEIT,0.1209,0.0000,0.0,0.8062,"[vans, vans, vans, vans, vans]",True
3,new balance-4.jpg,LIKELY_COUNTERFEIT,0.3956,0.0000,0.8,0.7706,"[newbalance, newbalance, vans, newbalance, new...",True
4,new balance-5.jpg,LIKELY_COUNTERFEIT,0.4803,0.0002,1.0,0.8681,"[newbalance, newbalance, newbalance, newbalanc...",True
5,nike-1.jpg,LIKELY_COUNTERFEIT,0.4794,0.0000,1.0,0.8624,"[nike, nike, nike, nike, nike]",True
6,nike-2.jpg,LIKELY_COUNTERFEIT,0.4129,0.0000,0.8,0.8863,"[nike, puma, nike, nike, nike]",True
7,nike-3.jpg,LIKELY_COUNTERFEIT,0.4785,0.0001,1.0,0.8565,"[nike, nike, nike, nike, nike]",True
8,nike-4.jpg,LIKELY_COUNTERFEIT,0.1017,0.0000,0.0,0.6781,"[puma, puma, puma, puma, puma]",True
9,nike-5.jpg,LIKELY_COUNTERFEIT,0.2011,0.0000,0.2,0.8742,"[puma, puma, puma, nike, puma]",True
